# Layer 3 — 慢機台下鑽

如果 device side 是主要瓶頸，按 device_id 聚合，找出最慢的 devices。
按 loc_1, loc_2, system_name, device_mode_name 做切面分析。

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

REPORTS_DIR = Path('../reports')
PARALLELISM = 4

df = pd.read_csv('../data/orders.csv')
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format='%Y/%m/%d %I:%M:%S %p')

# Load and merge anomaly flags
sys_flags = pd.read_csv('../data/system_anomaly_flags.csv')
usr_flags = pd.read_csv('../data/user_anomaly_flags.csv')
df = df.merge(sys_flags, on='order_id')
df = df.merge(usr_flags[['order_id', 'is_user_anomaly']], on='order_id')

normal = df[~(df['is_system_anomaly'] | df['is_user_anomaly'])].copy()
print(f"Normal orders: {len(normal)}")


Normal orders: 27033


## 1. Device Performance Ranking

In [2]:
# Aggregate device-level stats
device_perf = normal.groupby('device_id').agg(
    device_dur_median=('device_duration_avg_seconds', 'median'),
    device_dur_mean=('device_duration_avg_seconds', 'mean'),
    device_dur_p95=('device_duration_avg_seconds', lambda x: x.quantile(0.95)),
    order_count=('order_id', 'count'),
    avg_file_count=('file_count', 'mean'),
    total_dur_median=('total_duration_seconds', 'median'),
).reset_index()

# Merge location info
loc_info = df.groupby('device_id').agg(
    loc_1=('loc_1', 'first'),
    loc_2=('loc_2', 'first'),
    system_name=('system_name', 'first'),
    device_mode_name=('device_mode_name', 'first'),
).reset_index()
device_perf = device_perf.merge(loc_info, on='device_id')

# Top 10 slowest devices
top10_slow = device_perf.nlargest(10, 'device_dur_median')
print("Top 10 Slowest Devices (by median device_duration_avg):")
print(top10_slow[['device_id', 'device_dur_median', 'device_dur_mean', 'device_dur_p95',
                   'order_count', 'loc_1', 'system_name']].to_string(index=False))


Top 10 Slowest Devices (by median device_duration_avg):
device_id  device_dur_median  device_dur_mean  device_dur_p95  order_count loc_1 system_name
 DEV-0062               14.0        13.808824            21.0          136 FAB-B   SYS-ALPHA
 DEV-0006               13.0        13.360902            21.0          133 FAB-A    SYS-BETA
 DEV-0028               13.0        13.536585            21.0          123 FAB-C   SYS-GAMMA
 DEV-0035               13.0        13.266187            20.1          139 FAB-C   SYS-GAMMA
 DEV-0057               13.0        13.625000            20.0          136 FAB-A   SYS-GAMMA
 DEV-0163               13.0        13.091503            20.0          153 FAB-B    SYS-BETA
 DEV-0189               13.0        13.191176            21.0          136 FAB-A    SYS-BETA
 DEV-0070               12.5        13.000000            21.0          112 FAB-C   SYS-GAMMA
 DEV-0000                3.0         2.664179             5.0          134 FAB-C   SYS-GAMMA
 DEV-0001     

## 2. 圖表

In [3]:
# Chart 1: Device performance ranking (top 30)
top30 = device_perf.nlargest(30, 'device_dur_median')

fig, ax = plt.subplots(figsize=(14, 8))
colors = ['#e74c3c' if d in top10_slow['device_id'].values else '#3498db' for d in top30['device_id']]
bars = ax.barh(range(len(top30)), top30['device_dur_median'].values, color=colors)
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(top30['device_id'].values, fontsize=8)
ax.set_title('Device Performance Ranking - Top 30 Slowest (Red = Top 10)')
ax.set_xlabel('Median Device Duration Avg (seconds)')
ax.invert_yaxis()

# Add order count annotation
for i, (dur, cnt) in enumerate(zip(top30['device_dur_median'], top30['order_count'])):
    ax.text(dur + 0.5, i, f'n={cnt}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig(REPORTS_DIR / '3_device_ranking.png', dpi=150)
plt.show()
print("Saved: 3_device_ranking.png")


Saved: 3_device_ranking.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65795/1924681876.py:19: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [4]:
# Chart 2: Faceted analysis by loc_1, system_name
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# By loc_1
loc1_stats = normal.groupby('loc_1')['device_duration_avg_seconds'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False)
loc1_stats['median'].plot(kind='bar', ax=axes[0][0], color=sns.color_palette('Set2'))
axes[0][0].set_title('Median Device Duration by Location (loc_1)')
axes[0][0].set_ylabel('Median Duration (seconds)')
axes[0][0].tick_params(axis='x', rotation=0)
for i, (idx, row) in enumerate(loc1_stats.iterrows()):
    axes[0][0].text(i, row['median'] + 0.1, f'n={int(row["count"])}', ha='center', fontsize=8)

# By system_name
sys_stats = normal.groupby('system_name')['device_duration_avg_seconds'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False)
sys_stats['median'].plot(kind='bar', ax=axes[0][1], color=sns.color_palette('Set2'))
axes[0][1].set_title('Median Device Duration by System')
axes[0][1].set_ylabel('Median Duration (seconds)')
axes[0][1].tick_params(axis='x', rotation=0)
for i, (idx, row) in enumerate(sys_stats.iterrows()):
    axes[0][1].text(i, row['median'] + 0.1, f'n={int(row["count"])}', ha='center', fontsize=8)

# By loc_2 (exclude NaN)
loc2_data = normal.dropna(subset=['loc_2'])
loc2_stats = loc2_data.groupby('loc_2')['device_duration_avg_seconds'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False)
loc2_stats['median'].plot(kind='bar', ax=axes[1][0], color=sns.color_palette('Set2'))
axes[1][0].set_title('Median Device Duration by Location (loc_2)')
axes[1][0].set_ylabel('Median Duration (seconds)')
axes[1][0].tick_params(axis='x', rotation=45)
for i, (idx, row) in enumerate(loc2_stats.iterrows()):
    axes[1][0].text(i, row['median'] + 0.1, f'n={int(row["count"])}', ha='center', fontsize=8)

# By device_mode_name (exclude NaN)
mode_data = normal.dropna(subset=['device_mode_name'])
mode_stats = mode_data.groupby('device_mode_name')['device_duration_avg_seconds'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False)
mode_stats['median'].plot(kind='bar', ax=axes[1][1], color=sns.color_palette('Set2'))
axes[1][1].set_title('Median Device Duration by Device Mode')
axes[1][1].set_ylabel('Median Duration (seconds)')
axes[1][1].tick_params(axis='x', rotation=0)
for i, (idx, row) in enumerate(mode_stats.iterrows()):
    axes[1][1].text(i, row['median'] + 0.1, f'n={int(row["count"])}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(REPORTS_DIR / '3_facet_analysis.png', dpi=150)
plt.show()
print("Saved: 3_facet_analysis.png")


Saved: 3_facet_analysis.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65795/3901745007.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [5]:
# Chart 3: Top 10 slow devices - detailed breakdown
fig, ax = plt.subplots(figsize=(14, 6))
top10_orders = normal[normal['device_id'].isin(top10_slow['device_id'])]

# Phase breakdown for slow devices
top10_orders['est_device'] = top10_orders['device_duration_avg_seconds'] * top10_orders['file_count'] / PARALLELISM
top10_orders['est_db'] = top10_orders['db_duration_avg_seconds'] * top10_orders['file_count'] / PARALLELISM
top10_orders['est_inner'] = top10_orders['inner_processing_duration_avg_seconds'] * top10_orders['file_count'] / PARALLELISM
top10_orders['est_queue'] = top10_orders['queue_duration_seconds']

phase_by_device = top10_orders.groupby('device_id')[['est_queue', 'est_db', 'est_device', 'est_inner']].mean()
phase_by_device.columns = ['Queue', 'DB', 'Device', 'Inner']
phase_by_device = phase_by_device.sort_values('Device', ascending=True)

phase_by_device.plot(kind='barh', stacked=True, ax=ax,
                     color=['#f39c12', '#3498db', '#e74c3c', '#2ecc71'])
ax.set_title('Phase Breakdown for Top 10 Slowest Devices')
ax.set_xlabel('Mean Estimated Duration (seconds)')
ax.set_ylabel('Device ID')
ax.legend(title='Phase')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '3_slow_device_breakdown.png', dpi=150)
plt.show()
print("Saved: 3_slow_device_breakdown.png")


Saved: 3_slow_device_breakdown.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65795/3826717030.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top10_orders['est_device'] = top10_orders['device_duration_avg_seconds'] * top10_orders['file_count'] / PARALLELISM
/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65795/3826717030.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top10_orders['est_db'] = top10_orders['db_duration_avg_seconds'] * top10_orders['file_count'] / PARALLELISM
/var/folders/8w/j7kvkq4x17s8f3hv

## 3. Summary

In [6]:
print("=== Layer 3 Summary ===")
print(f"\nTop 10 Slowest Devices:")
for _, row in top10_slow.iterrows():
    print(f"  {row['device_id']}: median={row['device_dur_median']:.1f}s, "
          f"p95={row['device_dur_p95']:.1f}s, orders={row['order_count']}, "
          f"loc={row['loc_1']}/{row['loc_2']}, sys={row['system_name']}")

print(f"\nLocation breakdown:")
for loc, row in loc1_stats.iterrows():
    print(f"  {loc}: median={row['median']:.1f}s ({int(row['count'])} orders)")


=== Layer 3 Summary ===

Top 10 Slowest Devices:
  DEV-0062: median=14.0s, p95=21.0s, orders=136, loc=FAB-B/None, sys=SYS-ALPHA
  DEV-0006: median=13.0s, p95=21.0s, orders=133, loc=FAB-A/AREA-2, sys=SYS-BETA
  DEV-0028: median=13.0s, p95=21.0s, orders=123, loc=FAB-C/None, sys=SYS-GAMMA
  DEV-0035: median=13.0s, p95=20.1s, orders=139, loc=FAB-C/AREA-1, sys=SYS-GAMMA
  DEV-0057: median=13.0s, p95=20.0s, orders=136, loc=FAB-A/AREA-2, sys=SYS-GAMMA
  DEV-0163: median=13.0s, p95=20.0s, orders=153, loc=FAB-B/None, sys=SYS-BETA
  DEV-0189: median=13.0s, p95=21.0s, orders=136, loc=FAB-A/AREA-2, sys=SYS-BETA
  DEV-0070: median=12.5s, p95=21.0s, orders=112, loc=FAB-C/AREA-1, sys=SYS-GAMMA
  DEV-0000: median=3.0s, p95=5.0s, orders=134, loc=FAB-C/AREA-1, sys=SYS-GAMMA
  DEV-0001: median=3.0s, p95=5.0s, orders=152, loc=FAB-A/AREA-1, sys=SYS-ALPHA

Location breakdown:
  FAB-A: median=3.0s (7729 orders)
  FAB-B: median=3.0s (9425 orders)
  FAB-C: median=3.0s (9879 orders)
